# 🎬 Hybrid Movie Recommendation System using Content-Based and Collaborative Filtering

This project builds a movie recommendation system using two approaches:

- Content-Based Filtering (based on movie features)
- Collaborative Filtering (based on user ratings)

Both are combined into a hybrid model to improve recommendation quality.

In [2]:
import pandas as pd
import numpy as np
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Load Data

We load movie information and user ratings dataset.

In [3]:
movies = pd.read_csv(r"C:\Users\myesw\Downloads\jai matha\movies.csv")
ratings = pd.read_csv(r"C:\Users\myesw\Downloads\jai matha\ratings.csv")

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## 📊 Understanding the Data

The dataset contains:
- Movie information (title, genres)
- User ratings for different movies

We use movie metadata for content-based filtering  
and user ratings for collaborative filtering.

## Data Preparation

- Remove missing values  
- Extract year from movie title  
- Create a combined text feature for similarity

In [4]:
movies = movies.dropna()

movies['year'] = movies['title'].str.extract(r'\((\d{4})\)')
movies['title_clean'] = movies['title'].str.replace(r'\(\d{4}\)', '', regex=True)

movies['tags'] = movies['genres'].str.replace('|', ' ') + ' ' + movies['title_clean']

movies[['title', 'tags']].head()

,title,tags
0,Toy Story (1995),Adventure Animation Children Comedy Fantasy To...
1,Jumanji (1995),Adventure Children Fantasy Jumanji
2,Grumpier Old Men (1995),Comedy Romance Grumpier Old Men
3,Waiting to Exhale (1995),Comedy Drama Romance Waiting to Exhale
4,Father of the Bride Part II (1995),Comedy Father of the Bride Part II




## 🧹 Data Preparation

Before building the model:
- Removed missing values
- Extracted year from movie title
- Combined features into a single column

This helps in creating meaningful similarity between movies.

## Content-Based Filtering

Convert text into numerical vectors using TF-IDF  
Then compute similarity between movies.

In [5]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(movies['tags'])

content_similarity = cosine_similarity(tfidf_matrix)

content_similarity.shape

(9742, 9742)

## 🔍 Content-Based Filtering

In this approach, movies are recommended based on their features.

- Genres and titles are converted into vectors using TF-IDF
- Cosine similarity is used to measure similarity between movies

This method works well for finding similar content.

## Collaborative Filtering

Create user-item matrix from ratings  
Then compute similarity between movies based on user behavior.

In [6]:
user_item = ratings.pivot_table(index='userId', columns='movieId', values='rating').fillna(0)

movie_similarity_cf = cosine_similarity(user_item.T)

movie_similarity_cf.shape

(9724, 9724)

## 🤝 Collaborative Filtering

This method uses user behavior:

- A user-item matrix is created from ratings
- Similarity is computed based on how users rate movies

This helps capture patterns in user preferences.

## Hybrid Recommendation Logic

We combine both models:

Final Score = 0.6 × Content + 0.4 × Collaborative

This helps improve recommendation accuracy.

In [7]:
def recommend(movie_name):
    
    movie_name = movie_name.lower()
    movies['title_lower'] = movies['title'].str.lower()
    
    if movie_name not in movies['title_lower'].values:
        return "Movie not found"
    
    idx = movies[movies['title_lower'] == movie_name].index[0]
    
    content_scores = list(enumerate(content_similarity[idx]))
    
    try:
        cf_scores = list(enumerate(movie_similarity_cf[idx]))
    except:
        cf_scores = content_scores
    
    min_len = min(len(content_scores), len(cf_scores))
    
    hybrid_scores = []
    
    for i in range(min_len):
        score = (0.6 * content_scores[i][1]) + (0.4 * cf_scores[i][1])
        hybrid_scores.append((i, score))
    
    hybrid_scores = sorted(hybrid_scores, key=lambda x: x[1], reverse=True)
    
    result = []
    for i in hybrid_scores[1:6]:
        result.append(movies.iloc[i[0]].title)
    
    return result

## ⚙️ Hybrid Approach

Both methods have limitations individually.

- Content-based → limited diversity
- Collaborative → depends on user data

So, we combine both using a weighted approach:

Final Score = 0.6 × Content + 0.4 × Collaborative

This improves overall recommendation quality.

## Test the Model

Check if the recommendation system works.

## 🧪 Testing the System

We test the model using a sample movie  
to check if it generates meaningful recommendations.

In [8]:
print("Recommendations:")
print(recommend("Toy Story (1995)"))

Recommendations:
['Toy Story 3 (2010)', 'Toy Story 2 (1999)', 'Toy, The (1982)', 'Aladdin (1992)', 'Madagascar (2005)']


## ⚠️ Limitations

- Cold start problem for new users/movies
- Does not consider real-time user feedback
- Performance may slow down for very large datasets

## ✅ Conclusion

In this project, a hybrid movie recommendation system was built using both content-based and collaborative filtering techniques.

The system successfully generates relevant movie suggestions by combining movie features and user behavior patterns.

This approach improves recommendation quality compared to using a single method.

The project demonstrates how machine learning can be applied to real-world problems like personalization and content recommendation.

## 💡 Business Perspective

Recommendation systems are widely used in platforms like Netflix, Amazon, and Spotify.

This system can help:
- Increase user engagement
- Improve content discovery
- Personalize user experience

By recommending relevant items, businesses can retain users and improve overall satisfaction.

## Save Files for Deployment

Save processed data and similarity matrices for Streamlit app.

In [9]:
pickle.dump(movies, open("movies.pkl", "wb"))
pickle.dump(content_similarity, open("content.pkl", "wb"))
pickle.dump(movie_similarity_cf, open("cf.pkl", "wb"))

print("Saved successfully")

Saved successfully
